# ENG-209, Partie 1, Série 3

## NumPy

1. Importer le package numpy

In [1]:
import numpy as np
from numpy.typing import NDArray

2. Créez un array numpy de type `uint8`

    [[ 1, 16, 49],\
     [ 4, 25, 64],\
     [ 9, 36, 81]]
     
Utilisez uniquement les fonctions numpy vues cette semaine. Ne pas utiliser de liste ou de tuple Python, ne pas utiliser de boucle de contrôle (for-loop).

Astuce: Notez la particularité des valeurs dans le tableau.

In [2]:
m: NDArray[np.uint8] = np.arange(1,10,dtype=np.uint8).reshape(3,3).T ** 2
print(m)

[[ 1 16 49]
 [ 4 25 64]
 [ 9 36 81]]


3. Vérifiez les caractéristiques du tableau précédemment créé:
- nombre de dimensions
- nombre d'élements par dimensions
- type
- taille mémoire

In [3]:
print(m.ndim)
print(m.shape)
print(m.dtype)
print(f"{m.size * m.itemsize} bytes")

2
(3, 3)
uint8
9 bytes


4. Affichez la dernière colonne du tableau précédemment créé
* Bonus: afficher la première ligne, la première et dernière colonne, etc.

In [4]:
print(m[:,-1])

[49 64 81]


In [5]:
print(m[0])
print(m[:,[0,-1]])

[ 1 16 49]
[[ 1 49]
 [ 4 64]
 [ 9 81]]


5. Résoudre le système linéaire suivant:

$$\left \{
  \begin{array}{rcl}
    2x+y-z&=&1 \\
    x+2y+z&=&8 \\
    3x-y+2z&=&7
    \end{array}
\right.$$

Vérifiez la solution avec des opérations matricielles en numpy.

In [6]:
A: NDArray = np.array([[2,1,-1],[1,2,1],[3,-1,2]])
b: NDArray = np.array([1,8,7])
c: NDArray = np.linalg.solve(A,b)
print(c)

[1. 2. 3.]


In [7]:
print(np.allclose(A @ c - b, 0))

True


6. Écrivez une fonction qui remplace tous les éléments pairs d’un array par -3 et qui copie le résultat dans un nouvel array. L’array original ne doit pas être modifié.

In [8]:
x: NDArray = np.array([1,2,3,4,5,6,7,8,9,10,11,121])
y: NDArray = np.where(x % 2 == 0, -3, x)
print(y)

[  1  -3   3  -3   5  -3   7  -3   9  -3  11 121]


Nous allons maintenant comparer la performance de la multiplication de matrices en utilisant numpy et en utilisant des boucles imbriquées, comme nous l'avons fait en Python pur dans la série d'exercices 2.

Exécutez la cellule suivante pour définir le type `Matrix` et la fonction `mult` qui multiplie deux matrices en utilisant des boucles imbriquées, ainsi que la fonction `random_matrix` qui génère une matrice aléatoire. Cette cellule ne doit être modifiée; elle ne contient que des définitions de fonctions.

In [9]:
Matrix = list[list[float]]

def mult(mat1: Matrix, mat2: Matrix) -> Matrix:
    mat1_m, mat1_n = dim_m(mat1), dim_n(mat1)
    mat2_m, mat2_n = dim_m(mat2), dim_n(mat2)
    if mat1_m == 0 or mat2_m == 0 or mat1_n != mat2_m:
        return []
    return [
        [sum([mat1[i][k] * mat2[k][j] for k in range(mat2_m)]) for j in range(mat2_n)]
        for i in range(mat1_m)
    ]

def dim_m(mat: Matrix) -> int:
    return len(mat)

def dim_n(mat: Matrix) -> int:
    m = dim_m(mat)
    if m == 0:
        return 0
    n = len(mat[0])
    for i in range(1, m):
        if len(mat[i]) != n:
            return 0
    return n

def random_matrix(m: int, n: int) -> Matrix:
    return [[np.random.rand() for _ in range(n)] for _ in range(m)]

Faites tourner le code dans la cellule suivante. Il commence par `%%timeit`, qui est une directive spéciale de Jupyter Notebook pour mesurer le temps d'exécution de la cellule. La cellule génère deux matrices aléatoires de taille 100 $\times$ 100 et les multiplie sans utiliser numpy. Notez le temps d'exécution moyen affiché.

In [10]:
%%timeit
A = random_matrix(100, 100)
B = random_matrix(100, 100)
C = mult(A, B)

50.7 ms ± 153 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


7. Complétez la cellule suivante pour faire la même opération de multiplication de matrices, mais en utilisant numpy. Trouvez comment générer des matrices aléatoires et comment faire la multiplication matricielle en utilisant numpy. Notez le temps d'exécution moyen affiché et comparez-le avec celui de la cellule précédente. Quelle est l'ordre de grandeur du gain de performance?

In [11]:
%%timeit
A = np.random.rand(100, 100)
B = np.random.rand(100, 100)
C = A @ B

# L'ordre de grandeur du gain de performance de numpy sur ma machine est d'environ 1000.

87.5 μs ± 2.02 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Moindre carrés

Supposez qu'on ait le problème suivant

>Un drone mesure sa position 3 fois pendant un vol rectiligne. On sait qu’il se déplace selon une équation affine en 1D: $$x(t) = a t + b$$
> 
> où $t$ est le temps, $a$ la vitesse et $b$ la position initiale.
> 
> On observe les données suivantes:
> 
> $$\vec{t} = \begin{bmatrix} 0 \\ 1 \\ 2 \end{bmatrix}, \quad
\vec{x} = \begin{bmatrix} 2 \\ 5 \\ 8 \end{bmatrix}$$
> 
> Déterminez $a$ et $b$.

Nous pouvons calculer $a$ et $b$ via la méthode des moindres carrés. Pour l'instant, nous ne détaillons pas cette approche mais nous nous concentrons sur la notation et les opérations avec numpy; nous reparlerons plus en détail de la méthode des moindres carrés plus tard durant le cours.

Si l'on pose

$$\vec{w} = \begin{bmatrix} b \\ a \end{bmatrix},$$

comme vecteur des paramètres de l'équation, on peut écrire la relation suivante sous forme matricielle:

$$M \vec{w} = \vec{x},$$

avec $M$ qui est formé d'une colonne de 1 à côté du vecteur $\vec{t}$:

$$M = \begin{bmatrix}\vec{1} & \vec{t}\end{bmatrix} = \begin{bmatrix}1 & 0\\ 1 & 1\\ 1 & 2\end{bmatrix}.$$

Ensuite, on trouve la solution aux moindres carrés $\vec{w}^{\star}$ ainsi:

$$\vec{w}^{\star}=(M^T M)^{-1} M^T \vec{x}.$$

8. Complétez la cellule ci-dessous en utilisant les fonctions numpy pour définir les vecteurs $\vec{t}$ et $\vec{x}$, puis la matrice $M$ en utilisant $\vec{t}$, puis $M^T$, et finalement pour faire les opérations qui calculent $\vec{w}^{\star}$.

In [12]:
t = np.array([0, 1, 2])
x = np.array([2, 5, 8])
M = np.column_stack((np.ones_like(t), t))
M_T = M.T
w_star = np.linalg.inv(M_T @ M) @ M_T @ x
w_star

array([2., 3.])

## Classes 1

In [13]:
from dataclasses import dataclass

BEARINGS = ["N", "E", "S", "W"]

@dataclass
class Boat:
    x: int       # nombres entiers relatifs
    y: int       # idem
    bearing: str # "N", "E", "S" ou "W"

    def forward(self, distance: int) -> None:
        if self.bearing == "N":
            self.y += distance
        elif self.bearing == "S":
            self.y -= distance
        elif self.bearing == "E":
            self.x += distance
        elif self.bearing == "W":
            self.x -= distance

    def turn(self, clockwise: bool = True) -> None:
        current_bearing_index = BEARINGS.index(self.bearing)
        if clockwise:
            self.bearing = BEARINGS[(current_bearing_index + 1) % 4]
        else:
            self.bearing = BEARINGS[(current_bearing_index - 1) % 4]

    def exec(self, instructions: str) -> None:
        for instruction in instructions:
            if instruction == "F":
                self.forward(1)
            elif instruction == "L":
                self.turn(clockwise=False)
            elif instruction == "R":
                self.turn(clockwise=True)

In [14]:
def are_path_equivalent(instructions1: str, instructions2: str) -> bool:
    boat1 = Boat(0, 0, "N")
    boat2 = Boat(0, 0, "N")
    boat1.exec(instructions1)
    boat2.exec(instructions2)
    return boat1.x == boat2.x and boat1.y == boat2.y and boat1.bearing == boat2.bearing


In [15]:
def optimize(instructions: str) -> str:
    boat = Boat(0, 0, "N")
    boat.exec(instructions)

    if boat.x == 0 and boat.y == 0: # origine
        return ""
    if boat.x == 0 and boat.y > 0: # sur axe Y en haut
        return "F" * boat.y
    if boat.x == 0 and boat.y < 0: # sur axe Y en bas
        return "RR" + "F" * (-boat.y)
    if boat.x > 0 and boat.y == 0: # sur axe X à droite
        return "R" + "F" * boat.x
    if boat.x < 0 and boat.y == 0: # sur axe X à gauche
        return "L" + "F" * (-boat.x)
    if boat.x > 0 and boat.y > 0: # premier quadrant
        return "F" * boat.y + "R" + "F" * boat.x
    if boat.x < 0 and boat.y > 0: # deuxième quadrant
        return "F" * boat.y + "L" + "F" * (-boat.x)
    if boat.x < 0 and boat.y < 0: # troisième quadrant
        return "L" + "F" * (-boat.x) + "L" + "F" * (-boat.y)
    if boat.x > 0 and boat.y < 0: # quatrième quadrant
        return "R" + "F" * boat.x + "R" + "F" * (-boat.y)


In [16]:
tests = ["FFFRFFFFRF", "FLFLFLFLLRL", "RRFFF"]
for insts in tests:
    optimized = optimize(insts)
    print(f"Optimized version of '{insts}' is '{optimized}'")
    

Optimized version of 'FFFRFFFFRF' is 'FFRFFFF'
Optimized version of 'FLFLFLFLLRL' is ''
Optimized version of 'RRFFF' is 'RRFFF'


## Classes 2

In [17]:
@dataclass
class LifeTracker:
    life: int = 10
    num_regen: int = 0

    @property
    def is_dead(self) -> bool:
        return self.life <= 0
    
    def take_hit(self) -> None:
        if self.life <= 0:
            raise RuntimeError("already dead")
        self.life -= 1

    def regen(self) -> None:
        self.life = 10
        self.num_regen += 1

    def __str__(self) -> str:
        bar = (self.life // 2) * "X" + (self.life % 2) * "x" + ((10 - self.life) // 2) * "_"
        regen = "" if self.num_regen == 0 else f" ({self.num_regen})"
        return f"Life: {bar}{regen}"

In [18]:
life = LifeTracker()
print(life)
life.take_hit()
print(life)
life.take_hit()
print(life)
for _ in range(7):
    life.take_hit()
print(life)
print(life.is_dead)
life.take_hit()
print(life.is_dead)
life.regen()
print(life)

Life: XXXXX
Life: XXXXx
Life: XXXX_
Life: x____
False
True
Life: XXXXX (1)
